# DGX Spark Notebook 01 - Inference Smoke

Runs a short inference benchmark and writes machine-readable JSON output.

In [ ]:
from pathlib import Path
import json
import os
import subprocess

repo_root = Path.cwd()
if not (repo_root / 'scripts').exists() and (repo_root.parent / 'scripts').exists():
    repo_root = repo_root.parent
os.chdir(repo_root)

out_json = repo_root / 'artifacts/notebooks/inference-smoke.json'
out_json.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    'python3',
    'scripts/benchmark_inference.py',
    '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
    '--quantization', 'fp16',
    '--tokens', '64',
    '--runs', '1',
    '--device-map', 'cuda',
    '--output-json', str(out_json),
]
print('Running:', ' '.join(cmd))
proc = subprocess.run(cmd, check=True, text=True, capture_output=True)
print(proc.stdout[-4000:])

In [ ]:
payload = json.loads(out_json.read_text())
result = payload['results'][0]
assert result['tokens_per_sec'] > 0
assert result['gpu_memory_gb'] >= 0
result